middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for:-
Tracking agent behaviour with logging, analytics and debugging
transforming prompts, tool selection and output formatting
adding retries, fallbacks and early termination logic
applying rate limits, guardrails and PII detction

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
model=init_chat_model("google_genai:gemini-3.5-flash-lite")

###summarization middleware 
### automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. summarization is useful for long-running conversation that exceeded context window.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [3]:
### run witht a thread id
config={"configurable":{"thread_id":"test-1"} }


In [4]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='73d724f9-783a-4772-9b73-bfcfcd1edccd'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'El4KXAERTTIPKfruRKlldL7luyKprczM+2QMaI780kDLq7LvugumSONedUZnNhlB5Qv6sDlr52a/Ds+uOI/hAtMVJsWW/EpNbZBgunpEWUrf3jcEFmMTyy6BPcbh+ryM'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08fc5-b9f4-7a61-a36a-9db470f3c3a3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='73d724f9-783a-4772-9b73-bfcfcd1edccd'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'El4KXAERTTIPKfruRKl

In [5]:
### token size
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels in a city and return hotel information."""
    
    return f"""Hotels in {city}:

1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $180/night, business center
3. Budget Stay - 3 star, $75/night, free wifi
"""

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        )
    ],
)
config={"configurable":{"thread_id":"test-1"} }
def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages )
    return total_chars

In [6]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])

    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

Paris: ~672 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='8e95eef5-a766-4608-91ec-df15f11503e2'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_258322': 'El4KXAERTTIPU5MwhBBQoXD/56Uooqk1JZ5jrEXYUn0akma/SLo53M2CEdvVOnczYJ0GJd5DONy8KFbrJ5RLvx1zZV+vRedLWGta6iwMsTme75sxWs23bxPdKUsbHGln'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08fd5-6f0e-7ba0-8348-0c949d2a47dd-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_258322', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris:\n\n1. Grand Hotel - 5 star, $350/ni

In [7]:
###Fraction
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"


# LOW fraction for testing!
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),      # 0.2% = ~256 tokens
        ),
    ],
)
config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response["messages"])

Paris: ~101 tokens (0.0789%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='59a96922-7172-4ad8-ace5-0faaf0a07cea'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_339759': 'El4KXAERTTIPw3T3iMfe89nazWmZ+eZE7rsUyEyAQ/Rop3CF/OghFpaGc3h5orTK8Q1WXfoz9gYB/0TpJsogDl2rH10rbIqxRJP199eJ7x6OuqlKsMIuy1oeVkeDLgwD'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08fd9-146e-75f3-8a69-53ffc16c2d70-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_339759', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 16, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris: Grand Hotel $350, City Inn $180, B

HUMAN IN THE LOOP
PAUSE AGENT EXECUTION FOR HUMAN APPROVAL, EDITING OR REJECTING OF TOOL CALLS BEFORE THEY EXECUTE. HUMAN IN THE LOOP IS USEFUL FOR THE FOLLOWING:
high-stakes operations like financial transactions require human approval
long-running conversations where human feedback guides agent

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.tools import tool

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [15]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ],
)

In [16]:
config = {
    "configurable": {
        "thread_id": "test-approve"
    }
}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

print(result)

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0c400a3d-eadd-47b1-bf7c-b02bb83fbbb7'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_343343': 'El4KXAERTTIPHOJAbXqIEw7GJIOZ3RHhUcXGSSoRFjid7KzfFdjJivOB1+OPYz/U/fcg62sTiA7wh14jR98n5CVR0yufVJ0Z3ojM5HdJxKGVUU+HkRIv3JBYB1HJOQLi'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08ff2-949a-7373-a6ef-1fb3a41ea371-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'call_343343', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'outp

In [17]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0c400a3d-eadd-47b1-bf7c-b02bb83fbbb7'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_343343': 'El4KXAERTTIPHOJAbXqIEw7GJIOZ3RHhUcXGSSoRFjid7KzfFdjJivOB1+OPYz/U/fcg62sTiA7wh14jR98n5CVR0yufVJ0Z3ojM5HdJxKGVUU+HkRIv3JBYB1HJOQLi'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08ff2-949a-7373-a6ef-1fb3a41ea371-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'call_343343', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'ou

In [21]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'Email sent successfully to john@test.com.', 'extras': {'signature': 'El4KXAERTTIPmKBzA3waly310m94EB0g9IrSY5WUUcTSspwOSehfp8eAnp3qHYWCrE7juGasCNogdIdclL11RXVJs/+oWB3aXERU8g7Snr9hZ5NZJUXJCTQK7loAdpI3'}}]


In [ ]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ],
)

In [22]:
config = {
    "configurable": {
        "thread_id": "test-approve"
    }
}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

print(result)

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0c400a3d-eadd-47b1-bf7c-b02bb83fbbb7'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_343343': 'El4KXAERTTIPHOJAbXqIEw7GJIOZ3RHhUcXGSSoRFjid7KzfFdjJivOB1+OPYz/U/fcg62sTiA7wh14jR98n5CVR0yufVJ0Z3ojM5HdJxKGVUU+HkRIv3JBYB1HJOQLi'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08ff2-949a-7373-a6ef-1fb3a41ea371-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'call_343343', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'outp

In [23]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "The email send request was rejected and was not executed. Let me know if you'd like to do something else.", 'extras': {'signature': 'El4KXAERTTIPpdAR8L+VGxz/DJGp3khPJ4y5dlUnFYh3lfoOjVsvxSmvgqTmCs2gxvy0+dCK0JS3gqyRJhVeskD4GnnsCJVxxMZpYCT+2j2Z8g9AtnOW8Mn4/LhMY60S'}}]


In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0c400a3d-eadd-47b1-bf7c-b02bb83fbbb7'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_343343': 'El4KXAERTTIPHOJAbXqIEw7GJIOZ3RHhUcXGSSoRFjid7KzfFdjJivOB1+OPYz/U/fcg62sTiA7wh14jR98n5CVR0yufVJ0Z3ojM5HdJxKGVUU+HkRIv3JBYB1HJOQLi'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08ff2-949a-7373-a6ef-1fb3a41ea371-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'call_343343', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'ou

In [25]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ],
)

In [26]:
config = {
    "configurable": {
        "thread_id": "test-approve"
    }
}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to wrong@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

print(result)

{'messages': [HumanMessage(content="Send email to wrong@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='5a3572e7-80c1-4121-a8fc-6d4b64e77639'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "wrong@test.com", "subject": "Hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_255113': 'El4KXAERTTIPxUvM5VQYMKrGkh8YLPh0sNm1oLPe0uJoVIM/ELU3PryCPz7nMaF1CUrtupft8aC3XjHDeo4qN22q7+iADCB6+HH15VRyNbleHKgjExWmxxvkJMXCdsGc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08ff8-ff54-73a3-8af8-76f8324fd7e7-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_255113', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'o

In [27]:
result

{'messages': [HumanMessage(content="Send email to wrong@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='5a3572e7-80c1-4121-a8fc-6d4b64e77639'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "wrong@test.com", "subject": "Hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_255113': 'El4KXAERTTIPxUvM5VQYMKrGkh8YLPh0sNm1oLPe0uJoVIM/ELU3PryCPz7nMaF1CUrtupft8aC3XjHDeo4qN22q7+iADCB6+HH15VRyNbleHKgjExWmxxvkJMXCdsGc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08ff8-ff54-73a3-8af8-76f8324fd7e7-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_255113', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 

In [28]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",        # Tool name
                            "args": {                         # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

⏸️ Paused! Editing...


In [29]:
result

{'messages': [HumanMessage(content="Send email to wrong@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='5a3572e7-80c1-4121-a8fc-6d4b64e77639'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "wrong@test.com", "subject": "Hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_255113': 'El4KXAERTTIPxUvM5VQYMKrGkh8YLPh0sNm1oLPe0uJoVIM/ELU3PryCPz7nMaF1CUrtupft8aC3XjHDeo4qN22q7+iADCB6+HH15VRyNbleHKgjExWmxxvkJMXCdsGc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08ff8-ff54-73a3-8af8-76f8324fd7e7-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Corrected Subject', 'body': 'This was edited by human before sending'}, 'id': 'call_255113'}], invalid_tool_calls